# Experiment 013: 7B SFT-to-GRPO run, seed 2 (Colab)

This run uses the same 7B SFT-to-GRPO protocol and 500 GRPO updates as the runs
for seeds 0 and 1. Adding seed 2 completes the comparison across three seeds.
For seed 2, [experiment 022](022_colab_b7_smooth1000.ipynb) starts from the
merged supervised model saved here and uses the corresponding supervised adapter
for evaluation. The decision to run the later arms and the choice of their
settings were independent of this run’s accuracy.

Reuses the phase 1 base evals automatically (same experiment id, they skip
in seconds), pilot-locked exec, ladder off. ~4.5h, likely one session,
resume-safe throughout.

If the run OOMs: do NOT rerun as-is. Archive the failed directory into
failed_attempts following the registered OOM procedure, then restart from
step zero, full procedure in the 7b plan doc!

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 346.77   # from the resources panel
unit_rate = 6.77   # units per hour, the plan requires it per session
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '013_colab_7b_seed2', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260730_150158 | cu_before = 346.77
Mounted at /content/drive
closed previous session: 20260730_145903
saved open session: 20260730_150158
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 7b runs'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-ba846b5a-880d-d2f2-3390-9df5d1823a82)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 2551, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 2551 (delta 12), reused 28 (delta 10), pack-reused 2516 (from 1)
Receiving objects: 100% (2551/2551), 75.91 MiB | 18.43 MiB/s, done.
Resolving deltas: 100% (1631/1631), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')

# the package plan pins every arm to the recorded 013 environment, so the
# reference manifest is required and a missing one stops the session
import importlib, json, os
ref = 'results/013_COLAB_FULL_B7_sft_grpo_seed0/manifest.json'
assert os.path.exists(ref), 'reference manifest missing, pull the repo with results before training'
par = json.load(open(ref))['versions']
for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
    have = importlib.import_module(lib).__version__
    assert have == par[lib], f'{lib} {have} differs from the recorded {par[lib]}'
print('environment matches the recorded 013 versions')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 158.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.2 MB/s eta 0:00:00
deps installed
environment matches the recorded 013 versions


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 60, 'need >=60GB free on drive for the 7b phases'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260730_150158.json
drive free: 179 GB


## 6. Build the data splits

In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 20.2MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.63MB/s,  223kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.63MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 338kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 338kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 351630.98 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 263003.90 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 88.0MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 143MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 7. Preflight: the shared base evals from phase 1 must exist

In [8]:
import os
D = '/content/drive/MyDrive/inm363'
be = os.path.join(D, 'results', 'base_evals', '013_COLAB_FULL_B7')
for f in ('base_eval.json', 'base_logprob.json'):
    assert os.path.exists(os.path.join(be, f)), f'phase 1 base eval missing: {f}'
print('shared base evals present, the run will skip them')

shared base evals present, the run will skip them


## 8. Wiring check

In [9]:
!python scripts/run.py --config configs/013_colab_full_b7_seed2.yaml

env=COLAB mode=FULL scale=B7 condition=SFT_GRPO
model=Qwen/Qwen2.5-7B device=cuda stages=['base_eval', 'sft', 'sft_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=2
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/013_COLAB_FULL_B7_sft_grpo_seed2/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/013_COLAB_FULL_B7_sft_grpo_seed2/
local smoke wiring ok


## 9. The run

**If colab disconnects:** re-run this cell, finished stages skip.

In [ ]:
# both cost fields must be filled before the long run starts
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
# python loop with ! calls so the logs stream live, exit code checked so a
# failed run stops the session instead of hiding behind the next one
for cfg in ('013_colab_full_b7_seed2',):
    print(f'===== {cfg} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{cfg}.yaml --stage all
    if _exit_code != 0:
        raise SystemExit(f'{cfg} failed with exit code {_exit_code}, see the OOM protocol before retrying')

Streaming output truncated to the last 5000 lines.
│ │               │ Combine the   │               │              │           │ │
│ │               │ two           │               │              │           │ │
│ │               │ kettle-soup-… │               │              │           │ │
│ │               │ and you get   │               │              │           │ │
│ │               │ half a plane. │               │              │           │ │
│ │               │ There are 6   │               │              │           │ │
│ │               │ square units  │               │              │           │ │
│ │               │ in the        │               │              │           │ │
│ │               │ rectangle     │               │              │           │ │
│ │               │ defined by    │               │              │           │ │
│ │               │ the $x$ and   │               │              │           │ │
│ │               │ $y$           │               │       

## 10. Peek at the results

In [ ]:
import json, os
D = '/content/drive/MyDrive/inm363'
for tag in ('013_COLAB_FULL_B7_sft_grpo_seed0', '013_COLAB_FULL_B7_sft_grpo_seed1',
            '013_COLAB_FULL_B7_sft_grpo_seed2'):
    sm = os.path.join(D, 'results', tag, 'summary.json')
    if os.path.exists(sm):
        j = json.load(open(sm))
        print(f"{tag}: sft {j['sft_acc']}  grpo {j['grpo_acc']}")
    else:
        print(f'{tag}: no summary yet')

013_COLAB_FULL_B7_sft_grpo_seed0: sft {'easy': 0.8816837315130831, 'medium': 0.7569786535303776, 'hard': 0.5883838383838383}  grpo {'easy': 0.8930602957906713, 'medium': 0.7766830870279147, 'hard': 0.6388888888888888}
013_COLAB_FULL_B7_sft_grpo_seed1: sft {'easy': 0.8782707622298066, 'medium': 0.7536945812807881, 'hard': 0.5959595959595959}  grpo {'easy': 0.8919226393629124, 'medium': 0.7766830870279147, 'hard': 0.6565656565656566}
013_COLAB_FULL_B7_sft_grpo_seed2: sft {'easy': 0.888509670079636, 'medium': 0.7783251231527094, 'hard': 0.6363636363636364}  grpo {'easy': 0.8862343572241184, 'medium': 0.7980295566502463, 'hard': 0.6439393939393939}


## 11. Close the units record

In [ ]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 307.28    # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
